In [ ]:
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

# Find the repo root from wherever this notebook runs (fixes the relative-path bug from M0)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "meridian.duckdb"), read_only=True)

def q(sql: str) -> pl.DataFrame:
    return con.sql(sql).pl()

print(q("select count(*) as sales, min(sale_date) as first_day, max(sale_date) as last_day from main_marts.fct_sales"))

In [ ]:
kpi = q("""
    with orders as (                       -- an 'order' = one customer on one day
        select customer_id, sale_date,
               date_trunc('month', sale_date)::date as month,
               count(*) as units, sum(price_index) as revenue
        from main_marts.fct_sales
        group by all
    )
    select month,
           count(distinct customer_id)              as customers,
           count(*)                                 as orders,
           sum(units)                               as units,
           sum(revenue)                             as revenue,
           count(*) / count(distinct customer_id)   as frequency,     -- orders per customer
           sum(revenue) / count(*)                  as basket_value   -- revenue per order
    from orders
    where month between date '2018-10-01' and date '2020-08-01'   -- complete months only
    group by month
    order by month
""")
check = (kpi["customers"] * kpi["frequency"] * kpi["basket_value"] - kpi["revenue"]).abs().max()
print(f"identity check (should be ~0): {check:.10f}")
kpi.head()

In [ ]:
idx = kpi.select(
    "month",
    *[(pl.col(c) / pl.col(c).first() * 100).alias(c) for c in ["revenue", "customers", "frequency", "basket_value"]],
)
fig = px.line(
    idx.unpivot(index="month", variable_name="kpi", value_name="index"),
    x="month", y="index", color="kpi",
    title="Revenue and its 3 drivers, indexed (Oct 2018 = 100)",
)
fig.add_hline(y=100, line_dash="dot", line_color="grey")
fig.show()

In [ ]:
yoy = (
    kpi.with_columns(pl.col("month").dt.offset_by("-1y").alias("month_ly"))
    .join(kpi, left_on="month_ly", right_on="month", suffix="_ly")
    .select(
        "month",
        ((pl.col("revenue") / pl.col("revenue_ly")).log() * 100).alias("revenue_pct_log"),
        ((pl.col("customers") / pl.col("customers_ly")).log() * 100).alias("customers"),
        ((pl.col("frequency") / pl.col("frequency_ly")).log() * 100).alias("frequency"),
        ((pl.col("basket_value") / pl.col("basket_value_ly")).log() * 100).alias("basket_value"),
    )
)
fig = go.Figure()
for c in ["customers", "frequency", "basket_value"]:
    fig.add_bar(x=yoy["month"], y=yoy[c], name=c)
fig.add_scatter(x=yoy["month"], y=yoy["revenue_pct_log"], name="total revenue change",
                mode="lines+markers", line=dict(color="black"))
fig.update_layout(barmode="relative", title="YoY revenue change split by driver (log %, bars sum to the line)",
                  yaxis_title="YoY change, log %")
fig.show()
yoy

In [ ]:
ret = q("""
    with active as (
        select distinct customer_id, date_trunc('month', sale_date)::date as month
        from main_marts.fct_sales
    ),
    first_seen as (
        select customer_id, min(month) as cohort from active group by 1
    )
    select f.cohort,
           datediff('month', f.cohort, a.month)     as months_since,
           count(*)                                 as active_customers
    from active a join first_seen f using (customer_id)
    group by all
""")
size = ret.filter(pl.col("months_since") == 0).select("cohort", pl.col("active_customers").alias("cohort_size"))
grid = (
    ret.join(size, on="cohort")
    .with_columns((pl.col("active_customers") / pl.col("cohort_size") * 100).round(1).alias("retention_pct"))
    .filter((pl.col("months_since").is_between(1, 12)) & (pl.col("cohort") >= pl.date(2019, 1, 1)))
    .pivot(on="months_since", index="cohort", values="retention_pct", sort_columns=True)
    .sort("cohort")
)
fig = px.imshow(
    grid.drop("cohort").to_numpy(),
    x=[str(c) for c in grid.columns[1:]], y=[str(d) for d in grid["cohort"]],
    labels=dict(x="months since first purchase", y="cohort (first month seen)", color="% active"),
    color_continuous_scale="Blues", text_auto=True, aspect="auto",
    title="Monthly cohort retention, % of cohort buying again",
)
fig.show()

In [ ]:
grid.select("cohort", pl.col("1").alias("m1_pct"), pl.col("3").alias("m3_pct"))

In [ ]:
git add modules; git commit -m "M1: KPI tree + cohort retention"; git push